## 1. API structure

In [45]:
import requests
print("API structure")
print("="*20)

try:
    response = requests.get("http://localhost:8000/openapi.json")
    if response.status_code == 200:
        openapi_data = response.json()
        endpoints = list(openapi_data["paths"].keys())

        print(f"Total endpoints: {len(endpoints)}")
        print("\nAvailable endpoints:")
        for endpoint in sorted(endpoints):
            print(f"  • {endpoint}")
    else:
        print(f"Could not fetch API info: {response.status_code}")
except Exception as e:
    print(f"Error: {e}")

API structure
Total endpoints: 2

Available endpoints:
  • /api/v1/health
  • /api/v1/hybrid-search/


## 2. Test Ollama LLM

In [15]:
print("Ollama LLM test")
try:
    models_response = requests.get("http://localhost:11434/api/tags")
    if models_response.status_code == 200:
        models = models_response.json().get('models', [])
        print(f"Available models: {len(models)}")
        for model in models:
            print(f"  • {model}")
    else:
        print(f"Could not fetch models: {models_response.status_code}")
except Exception as e:
    print(f"Error: {e}")

Ollama LLM test
Available models: 1
  • {'name': 'llama3.2:latest', 'model': 'llama3.2:latest', 'modified_at': '2026-01-18T14:22:53.877786012Z', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M'}}


In [20]:
# Test simple generation

try:
    test_data = {
        "model": "llama3.2:latest",
        "prompt": "What is the capital of France?",
        "stream": False
    }

    response = requests.post(
        "http://localhost:11434/api/generate",
        json=test_data,
        timeout=30
    )
    if response.status_code == 200:
        result = response.json()
        answer = result.get("response", '').strip()
        print(f" LLM response: {answer}")
    else:
        print(f"Error: {response.status_code}")
except Exception as e:
    print(f"Error: {e}")

 LLM response: The capital of France is Paris.


## 3. Test search functionality

In [43]:
print("Test search functionality")
print("="*20)

search_query = "machine learning"
print(f"Searching for: {search_query}")

try:
    search_request = {
        "query": search_query,
        "use_hybrid": True,
        "size": 3
    }
    response = requests.post(
        "http://localhost:8000/api/v1/hybrid-search",
        json=search_request,
        timeout=30 
    )

    if response.status_code == 200:
        data = response.json()
        print(f"✓ Found {data['total']} results")
        print(f"✓ Search mode: {data['search_mode']}")
        
        if data['hits']:
            print("\nTop results:")
            for i, hit in enumerate(data['hits'][:3], 1):
                title = hit.get('title', 'Unknown')[:60]
                score = hit.get('score', 0)
                print(f"  {i}. {title}... (score: {score:.3f})")
        else:
            print("No results found")
    else:
        print(f"✗ Search failed: {response.status_code}")
        
except Exception as e:
    print(f"✗ Error: {e}")

Test search functionality
Searching for: machine learning
✓ Found 3 results
✓ Search mode: hybrid

Top results:
  1. LaViT: Aligning Latent Visual Thoughts for Multi-modal Reaso... (score: 0.031)
  2. What Understanding Means in AI-Laden Astronomy... (score: 0.029)
  3. A Sustainable AI Economy Needs Data Deals That Work for Gene... (score: 0.016)


## 6. Complete RAG pipeline test

In [68]:
import requests
import time

print("Complete RAG pipeline test")
print("="*20)

question = "Summarize machine learning papers"
print(f"Question: {question}")

start_time = time.time()

try:
    rag_request = {
        "query": question,
        "top_k": 1,
        "use_hybrid": True,
        "model": "llama3.2:latest",
    }

    response = requests.post(
        "http://localhost:8000/api/v1/ask",
        json=rag_request,
        timeout=300
    )

    response_time = time.time() - start_time

    if response.status_code == 200:
        data = response.json()
        print(f"\n✓ Success! ({response_time:.1f} seconds)")
        print(f"\nAnswer:")
        print("-" * 40)
        print(data['answer'])
        print("-" * 40)
        
        print(f"\nSources: {len(data.get('sources', []))} papers")
        print(f"Chunks used: {data.get('chunks_used', 0)}")
        print(f"Search mode: {data.get('search_mode', 'unknown')}")

    else:
        print(f"\n✗ Request failed: HTTP {response.status_code}")
        print(f"Response: {response.text[:200]}")
        
except Exception as e:
    print(f"\n✗ Error: {e}")


        
except Exception as e:
    print(f"✗ Error: {e}")

Complete RAG pipeline test
Question: Summarize machine learning papers

✓ Success! (117.0 seconds)

Answer:
----------------------------------------
Here's a summary of the provided paper excerpt:

The paper proposes a new Generative Adversarial Network (GAN) architecture called Semi-Supervised Swin-Inspired GAN (SSI-GAN) for neuronal spike classification. The authors aim to address the scarcity of labeled data problems in classifying mosquito neural signals, which is crucial for detecting arboviral diseases like Zika and dengue.

The proposed SSI-GAN consists of a shifted-window transformer discriminator and a generator that uses multi-head self-attention models. The discriminator learns to capture sparser high-frequency spike features, while the generator produces synthetic data that can be used to improve classification accuracy.

The authors train the model using semi-supervised learning with labeled, unlabeled, and synthetic data jointly improving classification. They use Bayesian

## 7. Complete RAG pipeline test - streaming

In [73]:
import time
import json 
print("Complete rag pipeline test (streaming)")
print("="*40)

question = "Summarize machine learning papers?"
print(f"Question: {question}")

start_time = time.time()

try:
    rag_request = {
        "query": question,
        "top_k": 1,
        "use_hybrid": True,
        "model": "llama3.2:latest"
    }

    response = requests.post("http://localhost:8000/api/v1/stream", json=rag_request, stream=True, timeout=300)

    if response.status_code == 200:
        full_answer = ""
        sources = []
        chunks_used = 0
        search_mode = "unknown"
        first_chunk_time = None

        print(f"\nStreaming response...")
        for line in response.iter_lines():
            if line:
                line_str = line.decode("utf-8")
                if line_str.startswith("data: "):
                    try:
                        data = json.loads(line_str[6:])
                        if 'sources' in data:
                            sources = data['sources']
                            chunks_used = data.get("chunks_used", 0)
                            search_mode = data.get('search_mode', 'unknown')

                        if 'chunk' in data:
                            if first_chunk_time is None:
                                first_chunk_time = time.time() - start_time
                                print(f"First response in: {first_chunk_time:.2f} seconds")

                            chunk_text = data['chunk']
                            full_answer += chunk_text
                            print(chunk_text, end='', flush=True)

                        if data.get('done', False):
                            break
                    except json.JSONDecodeError as e:
                        print(f"JSON decode error: {e}")
                        continue
        response_time = time.time() - start_time
        print("\n" + "-" * 40)
        print(f"\n✓ Complete! (Total: {response_time:.1f} seconds)")
        
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources[:2], 1):
                print(f"  {i}. {source}")
        print(f"Chunks used: {chunks_used}")
        print(f"Search mode: {search_mode}")

    else:
        print(f"\n✗ Request failed: HTTP {response.status_code}")
        print(f"Response: {response.text[:200]}")
        
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

Complete rag pipeline test (streaming)
Question: Summarize machine learning papers?

Streaming response...
First response in: 51.39 seconds
To  summarize  machine  learning  papers ,  let 's  take  a  look  at  what  the  authors  of  " SSI -G AN :  Semi -S up ervised  Sw in -In sp ired  Gener ative  Ad vers arial  Networks  for  Ne ur onal  Spike  Classification "  have  to  say  about  it .  According  to  the  paper ,  the  main  goal  is  to  develop  a  new  Gener ative  Ad vers arial  Network  ( GAN )  architecture  that  can  classify  neuronal  spike  patterns  with  minimal  labeled  data .

 The  authors  propose  a  semi -sup ervised  approach  that  uses  both  labeled  and  unl abeled  data  to  improve  classification  accuracy .  They  use  a  Sw in -inspired  discriminator  and  a  transformer -based  generator  to  capture  high -frequency  spike  features  in  the  signals .  The  network  is  trained  using  Bayesian  Opt una  for  hyper parameter  optimization  and 